In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import random
import numpy as np
import torch
import torch.nn as nn
from torchinfo import summary

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
import shutil
import torch.nn.functional as F
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
import scipy.io
from copy import deepcopy
import copy
import random
import numpy as np
import pandas as pd
from pathlib import Path
import math
import pickle

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend

from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.metrics import recall_score, precision_score, cohen_kappa_score

import json
import hashlib

/home/usuario-utp/anaconda3/envs/yessica/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

class Conv2dSame(nn.Module):
    """
    Conv2D con padding 'same' real, incluyendo kernels pares como 64 o 16.
    """
    def __init__(self, in_channels, out_channels, kernel_size, bias=False, groups=1):
        super().__init__()

        if isinstance(kernel_size, int):
            kh, kw = kernel_size, kernel_size
        else:
            kh, kw = kernel_size

        pad_h = kh - 1
        pad_w = kw - 1

        pad_top = pad_h // 2
        pad_bottom = pad_h - pad_top
        pad_left = pad_w // 2
        pad_right = pad_w - pad_left

        self.pad = nn.ZeroPad2d((pad_left, pad_right, pad_top, pad_bottom))
        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=(kh, kw),
            padding=0,
            bias=bias,
            groups=groups,
        )

    def forward(self, x):
        x = self.pad(x)
        return self.conv(x)


class EEGNetTorch(nn.Module):
    def __init__(
        self,
        nb_classes=2,
        Chans=19,
        Samples=512,
        dropoutRate=0.5,
        kernLength=64,
        F1=8,
        D=2,
        F2=None,
        norm_rate=0.25,
        dropoutType="Dropout",
        bn_eps=1e-3,
        bn_momentum=0.01,  # PyTorch momentum equivalente aproximado a Keras momentum=0.99
        return_logits=False,
    ):
        super().__init__()

        if F2 is None:
            F2 = F1 * D

        self.nb_classes = nb_classes
        self.Chans = Chans
        self.Samples = Samples
        self.dropoutRate = dropoutRate
        self.kernLength = kernLength
        self.F1 = F1
        self.D = D
        self.F2 = F2
        self.norm_rate = norm_rate
        self.return_logits = return_logits

        if dropoutType == "SpatialDropout2D":
            DropLayer = nn.Dropout2d
        elif dropoutType == "Dropout":
            DropLayer = nn.Dropout
        else:
            raise ValueError("dropoutType debe ser 'Dropout' o 'SpatialDropout2D'.")

        # Entrada: (B, Chans, Samples)
        # Se convierte a: (B, 1, Chans, Samples)

        # =====================================================
        # Block 1: Conv2D temporal
        # =====================================================
        self.conv_temporal = Conv2dSame(
            in_channels=1,
            out_channels=F1,
            kernel_size=(1, kernLength),
            bias=False,
        )

        self.bn1 = nn.BatchNorm2d(
            F1,
            eps=bn_eps,
            momentum=bn_momentum,
        )

        # =====================================================
        # Depthwise spatial convolution
        # Equivalente a DepthwiseConv2D((Chans, 1), depth_multiplier=D)
        # =====================================================
        self.depthwise_spatial = nn.Conv2d(
            in_channels=F1,
            out_channels=F1 * D,
            kernel_size=(Chans, 1),
            groups=F1,
            bias=False,
        )

        self.bn2 = nn.BatchNorm2d(
            F1 * D,
            eps=bn_eps,
            momentum=bn_momentum,
        )

        self.pool1 = nn.AvgPool2d(kernel_size=(1, 4))
        self.dropout1 = DropLayer(dropoutRate)

        # =====================================================
        # Block 2: SeparableConv2D
        # Depthwise temporal + pointwise 1x1
        # =====================================================
        self.separable_depthwise = Conv2dSame(
            in_channels=F1 * D,
            out_channels=F1 * D,
            kernel_size=(1, 16),
            bias=False,
            groups=F1 * D,
        )

        self.separable_pointwise = nn.Conv2d(
            in_channels=F1 * D,
            out_channels=F2,
            kernel_size=(1, 1),
            bias=False,
        )

        self.bn3 = nn.BatchNorm2d(
            F2,
            eps=bn_eps,
            momentum=bn_momentum,
        )

        self.pool2 = nn.AvgPool2d(kernel_size=(1, 8))
        self.dropout2 = DropLayer(dropoutRate)

        # Después de pool1 /4 y pool2 /8: Samples / 32
        samples_after_pool = Samples // 32
        flatten_dim = F2 * samples_after_pool

        self.classifier = nn.Linear(flatten_dim, nb_classes)

    def forward(self, x):
        """
        x: (B, Chans, Samples)
        """

        # (B, Chans, Samples) -> (B, 1, Chans, Samples)
        x = x.unsqueeze(1)

        # Block 1
        x = self.conv_temporal(x)
        x = self.bn1(x)

        x = self.depthwise_spatial(x)
        x = self.bn2(x)
        x = F.elu(x)

        x = self.pool1(x)
        x = self.dropout1(x)

        # Block 2 separable
        x = self.separable_depthwise(x)
        x = self.separable_pointwise(x)
        x = self.bn3(x)
        x = F.elu(x)

        x = self.pool2(x)
        x = self.dropout2(x)

        flat = torch.flatten(x, start_dim=1)

        logits = self.classifier(flat)
        probs = torch.softmax(logits, dim=1)

        out = {
            "out_activation": probs,
        }

        if self.return_logits:
            out["logits"] = logits

        return out

    @torch.no_grad()
    def apply_constraints(self):
        """
        Imita:
        - depthwise_constraint = max_norm(1.)
        - kernel_constraint = max_norm(norm_rate) en Dense final.
        """
        apply_max_norm_to_conv2d_out_channels(
            self.depthwise_spatial,
            max_value=1.0,
        )

        apply_max_norm_to_linear(
            self.classifier,
            max_value=self.norm_rate,
        )


@torch.no_grad()
def apply_max_norm_to_linear(linear_layer, max_value=0.25, eps=1e-8):
    W = linear_layer.weight.data
    norms = W.norm(2, dim=1, keepdim=True).clamp_min(eps)
    desired = norms.clamp(max=max_value)
    W.mul_(desired / norms)


@torch.no_grad()
def apply_max_norm_to_conv2d_out_channels(conv_layer, max_value=1.0, eps=1e-8):
    """
    Restringe la norma por filtro de salida.
    Sirve para aproximar depthwise_constraint=max_norm(1.).
    """
    W = conv_layer.weight.data
    W_flat = W.view(W.shape[0], -1)
    norms = W_flat.norm(2, dim=1, keepdim=True).clamp_min(eps)
    desired = norms.clamp(max=max_value)
    W_flat.mul_(desired / norms)

In [3]:
# =========================================================
# Loss para EEGNet
# =========================================================

class EEGNetClassificationLoss(nn.Module):
    """
    Loss para EEGNet usando y_true one-hot y y_pred como probabilidades softmax.
    """
    def __init__(self, eps=1e-7):
        super().__init__()
        self.eps = eps

    def forward(self, outputs, y_true):
        y_pred = outputs["out_activation"]
        y_pred = torch.clamp(y_pred, self.eps, 1.0 - self.eps)

        loss = -torch.sum(y_true * torch.log(y_pred), dim=1)
        return loss.mean()

In [4]:

class Conv2dSame(nn.Module):
    """
    Conv2D con padding 'same' real, incluyendo kernels pares como 64 o 16.
    """
    def __init__(self, in_channels, out_channels, kernel_size, bias=False, groups=1):
        super().__init__()

        if isinstance(kernel_size, int):
            kh, kw = kernel_size, kernel_size
        else:
            kh, kw = kernel_size

        pad_h = kh - 1
        pad_w = kw - 1

        pad_top = pad_h // 2
        pad_bottom = pad_h - pad_top
        pad_left = pad_w // 2
        pad_right = pad_w - pad_left

        self.pad = nn.ZeroPad2d((pad_left, pad_right, pad_top, pad_bottom))

        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=(kh, kw),
            padding=0,
            bias=bias,
            groups=groups,
        )

    def forward(self, x):
        x = self.pad(x)
        return self.conv(x)


class EEGNetTorch(nn.Module):
    def __init__(
        self,
        nb_classes=2,
        Chans=19,
        Samples=512,
        dropoutRate=0.5,
        kernLength=64,
        F1=8,
        D=2,
        F2=None,
        norm_rate=0.25,
        dropoutType="Dropout",
        bn_eps=1e-3,
        bn_momentum=0.01,
        return_logits=False,
    ):
        super().__init__()

        if F2 is None:
            F2 = F1 * D

        self.nb_classes = nb_classes
        self.Chans = Chans
        self.Samples = Samples
        self.dropoutRate = dropoutRate
        self.kernLength = kernLength
        self.F1 = F1
        self.D = D
        self.F2 = F2
        self.norm_rate = norm_rate
        self.return_logits = return_logits

        if dropoutType == "SpatialDropout2D":
            DropLayer = nn.Dropout2d
        elif dropoutType == "Dropout":
            DropLayer = nn.Dropout
        else:
            raise ValueError("dropoutType debe ser 'Dropout' o 'SpatialDropout2D'.")

        # Entrada original: (B, Chans, Samples)
        # Se convierte en forward a: (B, 1, Chans, Samples)

        # =====================================================
        # Bloque 1: Conv2D temporal
        # =====================================================
        self.conv_temporal = Conv2dSame(
            in_channels=1,
            out_channels=F1,
            kernel_size=(1, kernLength),
            bias=False,
        )

        self.bn1 = nn.BatchNorm2d(
            F1,
            eps=bn_eps,
            momentum=bn_momentum,
        )

        # =====================================================
        # Depthwise spatial convolution
        # Equivalente a DepthwiseConv2D((Chans, 1), depth_multiplier=D)
        # =====================================================
        self.depthwise_spatial = nn.Conv2d(
            in_channels=F1,
            out_channels=F1 * D,
            kernel_size=(Chans, 1),
            groups=F1,
            bias=False,
        )

        self.bn2 = nn.BatchNorm2d(
            F1 * D,
            eps=bn_eps,
            momentum=bn_momentum,
        )

        self.pool1 = nn.AvgPool2d(kernel_size=(1, 4))
        self.dropout1 = DropLayer(dropoutRate)

        # =====================================================
        # Bloque 2: SeparableConv2D
        # Depthwise temporal + pointwise 1x1
        # =====================================================
        self.separable_depthwise = Conv2dSame(
            in_channels=F1 * D,
            out_channels=F1 * D,
            kernel_size=(1, 16),
            bias=False,
            groups=F1 * D,
        )

        self.separable_pointwise = nn.Conv2d(
            in_channels=F1 * D,
            out_channels=F2,
            kernel_size=(1, 1),
            bias=False,
        )

        self.bn3 = nn.BatchNorm2d(
            F2,
            eps=bn_eps,
            momentum=bn_momentum,
        )

        self.pool2 = nn.AvgPool2d(kernel_size=(1, 8))
        self.dropout2 = DropLayer(dropoutRate)

        # 512 -> /4 -> /8 = 16
        samples_after_pool = Samples // 32
        flatten_dim = F2 * samples_after_pool

        self.classifier = nn.Linear(flatten_dim, nb_classes)

    def forward(self, x):
        """
        x: (B, Chans, Samples)
        """

        # (B, Chans, Samples) -> (B, 1, Chans, Samples)
        x = x.unsqueeze(1)

        # Bloque 1
        x = self.conv_temporal(x)
        x = self.bn1(x)

        x = self.depthwise_spatial(x)
        x = self.bn2(x)
        x = F.elu(x)

        x = self.pool1(x)
        x = self.dropout1(x)

        # Bloque 2
        x = self.separable_depthwise(x)
        x = self.separable_pointwise(x)
        x = self.bn3(x)
        x = F.elu(x)

        x = self.pool2(x)
        x = self.dropout2(x)

        flat = torch.flatten(x, start_dim=1)

        logits = self.classifier(flat)
        probs = torch.softmax(logits, dim=1)

        out = {
            "out_activation": probs,
        }

        if self.return_logits:
            out["logits"] = logits

        return out

    @torch.no_grad()
    def apply_constraints(self):
        """
        Imita:
        - depthwise_constraint = max_norm(1.)
        - kernel_constraint = max_norm(norm_rate) en Dense final.
        """
        apply_max_norm_to_conv2d_out_channels(
            self.depthwise_spatial,
            max_value=1.0,
        )

        apply_max_norm_to_linear(
            self.classifier,
            max_value=self.norm_rate,
        )


@torch.no_grad()
def apply_max_norm_to_linear(linear_layer, max_value=0.25, eps=1e-8):
    W = linear_layer.weight.data
    norms = W.norm(2, dim=1, keepdim=True).clamp_min(eps)
    desired = norms.clamp(max=max_value)
    W.mul_(desired / norms)


@torch.no_grad()
def apply_max_norm_to_conv2d_out_channels(conv_layer, max_value=1.0, eps=1e-8):
    """
    Restringe la norma por filtro de salida.
    Aproxima depthwise_constraint=max_norm(1.).
    """
    W = conv_layer.weight.data
    W_flat = W.view(W.shape[0], -1)

    norms = W_flat.norm(2, dim=1, keepdim=True).clamp_min(eps)
    desired = norms.clamp(max=max_value)

    W_flat.mul_(desired / norms)

In [5]:
# =========================================================
# DEVICE
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================================================
# WRAPPER PARA QUE SUMMARY VEA SOLO LA SALIDA PRINCIPAL
# =========================================================
class SummaryWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        out = self.model(x)
        return out["out_activation"]   # (B, nb_classes)

# =========================================================
# CREAR MODELO EEGNet
# =========================================================
model = EEGNetTorch(
    nb_classes=2,
    Chans=19,
    Samples=512,
    dropoutRate=0.5,
    kernLength=32,
    F1=8,
    D=2,
    F2=16,
    norm_rate=0.25,
    dropoutType="Dropout",
).to(device)

wrapped_model = SummaryWrapper(model).to(device)

# =========================================================
# SUMMARY TIPO TENSORFLOW
# =========================================================
summary(
    wrapped_model,
    input_size=(1, 19, 512),   # (batch, chans, samples)
    depth=4,
    col_names=("input_size", "output_size", "num_params", "trainable"),
    device=device,
)

Using device: cuda


Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Trainable
SummaryWrapper                           [1, 19, 512]              [1, 2]                    --                        True
├─EEGNetTorch: 1-1                       [1, 19, 512]              [1, 2]                    --                        True
│    └─Conv2dSame: 2-1                   [1, 1, 19, 512]           [1, 8, 19, 512]           --                        True
│    │    └─ZeroPad2d: 3-1               [1, 1, 19, 512]           [1, 1, 19, 543]           --                        --
│    │    └─Conv2d: 3-2                  [1, 1, 19, 543]           [1, 8, 19, 512]           256                       True
│    └─BatchNorm2d: 2-2                  [1, 8, 19, 512]           [1, 8, 19, 512]           16                        True
│    └─Conv2d: 2-3                       [1, 8, 19, 512]           [1, 16, 1, 512]           304                       True
│    

In [6]:
def segmentar_senales(db, labels):
    """
    Divide las señales EEG en segmentos de 512 instantes con un traslape del 50%.

    Args:
        db (dict): Diccionario donde las claves son los nombres de los sujetos y los valores
                   son matrices de forma CxT_i (C = canales, T_i = tiempo).
        labels (array): Etiquetas por sujeto en formato 0/1.

    Returns:
        tuple:
            - segmentos: array de segmentos
            - y: array de etiquetas 0/1 por segmento
            - sbjs: lista de sujetos por segmento
            - window_ids: lista con identificador de ventana por segmento
    """
    segmento_tamano = 512
    paso = int(segmento_tamano * 0.5)  # 50% overlap
    i = 0

    segmentos = []
    y = []
    sbjs = []
    window_ids = []

    for sujeto, senal in db.items():
        C, T = senal.shape
        window_count = 1

        for inicio in range(0, T - segmento_tamano + 1, paso):
            segmento = senal[:, inicio:inicio + segmento_tamano]
            segmentos.append(segmento)
            y.append(labels[i])
            sbjs.append(sujeto)
            window_ids.append(f"Window {window_count}")
            window_count += 1

        i += 1

    return np.array(segmentos, dtype=np.float32), np.array(y, dtype=np.int64), sbjs, window_ids


def get_segmented_data():
    """
    Carga la base de datos y devuelve los segmentos listos para T-GARNet.
    """
    ruta_carpeta_TDAH = '/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/ieee/ADHD_group'
    ruta_carpeta_control = '/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/ieee/Control_group'

    sujetos_TDAH = sorted([
        archivo[:-4]
        for archivo in os.listdir(ruta_carpeta_TDAH)
        if archivo.endswith(".mat")
    ])

    sujetos_control = sorted([
        archivo[:-4]
        for archivo in os.listdir(ruta_carpeta_control)
        if archivo.endswith(".mat")
    ])

    # =========================================================
    # ELIMINAR EXPLÍCITAMENTE EL SUJETO QUE NO ESTÁ EN folds.pkl
    # =========================================================
    sujeto_a_eliminar = "v36p"

    if sujeto_a_eliminar in sujetos_TDAH:
        sujetos_TDAH.remove(sujeto_a_eliminar)
        print(f"Sujeto eliminado explícitamente: {sujeto_a_eliminar}")
    else:
        print(f"ADVERTENCIA: {sujeto_a_eliminar} no está en sujetos_TDAH")

    print("Sujetos TDAH encontrados:", len(sujetos_TDAH))
    print("Sujetos Control encontrados:", len(sujetos_control))
    print("Sujetos totales encontrados:", len(sujetos_TDAH) + len(sujetos_control))

    diagnostico = {}

    for sbj in sujetos_TDAH:
        diagnostico[sbj] = 1

    for sbj in sujetos_control:
        diagnostico[sbj] = 0

    eeg_tdah = {}
    for sbj in sujetos_TDAH:
        mat_file_path = ruta_carpeta_TDAH + '/' + sbj + '.mat'
        data = scipy.io.loadmat(mat_file_path)
        columna = list(data.keys())[-1]
        eeg_tdah[sbj] = data[columna].T

    eeg_control = {}
    for sbj in sujetos_control:
        mat_file_path = ruta_carpeta_control + '/' + sbj + '.mat'
        data = scipy.io.loadmat(mat_file_path)
        columna = list(data.keys())[-1]
        eeg_control[sbj] = data[columna].T

    db = eeg_control | eeg_tdah

    zeros = np.zeros(len(eeg_control))
    ones = np.ones(len(eeg_tdah))
    labels = np.hstack((zeros, ones))

    X, y, sbjs, window_ids = segmentar_senales(db, labels)

    y = np.eye(2, dtype=np.float32)[y]

    return X.astype(np.float32), y.astype(np.float32), sbjs, window_ids

In [7]:
# =========================================================
# CONFIGURACIÓN FIJA EEGNet
# =========================================================

model_name = "EEGNet"

model_args = {
    "Chans": 19,
    "Samples": 512,
    "nb_classes": 2,
    "dropoutRate": 0.5,
    "kernLength": 32,
    "F1": 8,
    "D": 2,
    "F2": 16,
    "norm_rate": 0.25,
    "dropoutType": "Dropout",
}

compile_cfg = {
    "learning_rate": 1e-2,
}

In [8]:
# =========================================================
# SEED
# =========================================================

def set_seed(seed=42, deterministic=True):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass

In [9]:
# =========================================================
# CONSTRUCTOR DEL MODELO
# =========================================================

def build_eeg_model(model_name, **kwargs):
    model_name = model_name.lower()

    if model_name != "eegnet":
        raise ValueError("Esta versión está preparada para model_name='EEGNet'.")

    # Importante: usa la semilla que le mande la rutina: seed + fold
    set_seed(kwargs.get("seed", 42))

    return EEGNetTorch(
        nb_classes=kwargs["nb_classes"],
        Chans=kwargs["Chans"],
        Samples=kwargs["Samples"],
        dropoutRate=kwargs.get("dropoutRate", 0.5),
        kernLength=kwargs.get("kernLength", 32),
        F1=kwargs.get("F1", 8),
        D=kwargs.get("D", 2),
        F2=kwargs.get("F2", 16),
        norm_rate=kwargs.get("norm_rate", 0.25),
        dropoutType=kwargs.get("dropoutType", "Dropout"),
        return_logits=False,
    )

In [10]:
# =========================================================
# CONFIGURACIÓN DE ENTRENAMIENTO / COMPILACIÓN
# =========================================================

def build_compile_config(model_name, **kwargs):
    model_name = model_name.lower()

    if model_name != "eegnet":
        raise ValueError("Esta versión está preparada para model_name='EEGNet'.")

    compile_args = {
        "optimizer_name": "adam",
        "optimizer_params": {
            "lr": kwargs.get("learning_rate", 1e-2),
        },
        "scheduler_name": None,
        "scheduler_params": {},
        "loss_fn": EEGNetClassificationLoss(),
        "loss_weights": None,
    }

    callbacks = []

    return compile_args, callbacks

In [11]:
# =========================================================
# UTILIDADES PARA ETIQUETAS
# =========================================================

def ensure_onehot_labels(y):
    y = np.asarray(y)

    # Ya viene one-hot [N, 2]
    if y.ndim == 2 and y.shape[1] == 2:
        return y.astype(np.float32)

    # Caso [N]
    if y.ndim == 1:
        y = y.astype(np.int64)
        return np.eye(2, dtype=np.float32)[y]

    # Caso [N, 1]
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.squeeze(1).astype(np.int64)
        return np.eye(2, dtype=np.float32)[y]

    raise ValueError(f"Formato de y no soportado para one-hot: shape={y.shape}")


def onehot_to_binary_labels(y):
    y = np.asarray(y)

    if y.ndim == 1:
        return y.astype(np.int64)

    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1).astype(np.int64)

    if y.ndim == 2 and y.shape[1] == 2:
        return np.argmax(y, axis=1).astype(np.int64)

    raise ValueError(f"Formato de y no soportado: shape={y.shape}")


# =========================================================
# OPTIMIZER / SCHEDULER
# =========================================================

def _build_optimizer_and_scheduler(model, compile_args):
    optimizer_name = compile_args["optimizer_name"].lower()
    optimizer_params = deepcopy(compile_args["optimizer_params"])

    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), **optimizer_params)
    else:
        raise ValueError(f"Optimizador no soportado: {optimizer_name}")

    scheduler_name = compile_args.get("scheduler_name", None)
    scheduler = None

    if scheduler_name is not None:
        scheduler_name = scheduler_name.lower()
        scheduler_params = deepcopy(compile_args.get("scheduler_params", {}))

        if scheduler_name == "reducelronplateau":
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, **scheduler_params
            )
        else:
            raise ValueError(f"Scheduler no soportado: {scheduler_name}")

    return optimizer, scheduler


# =========================================================
# UNA ÉPOCA DE ENTRENAMIENTO
# =========================================================

def _run_epoch_pytorch(model, loader, optimizer, loss_fn, device):
    model.train()
    running_loss = 0.0
    n_samples = 0

    for xb, yb in loader:
        xb = xb.to(device=device, dtype=torch.float32)
        yb = yb.to(device=device, dtype=torch.float32)

        optimizer.zero_grad()

        outputs = model(xb)
        loss = loss_fn(outputs, yb)

        loss.backward()
        optimizer.step()

        # Restricciones tipo Keras:
        # - depthwise max_norm(1.)
        # - dense final max_norm(norm_rate)
        if hasattr(model, "apply_constraints"):
            model.apply_constraints()
        elif hasattr(model, "classifier") and hasattr(model, "norm_rate"):
            apply_max_norm_to_linear(model.classifier, max_value=model.norm_rate)

        batch_size_curr = xb.size(0)
        running_loss += loss.item() * batch_size_curr
        n_samples += batch_size_curr

    return running_loss / max(n_samples, 1)


# =========================================================
# EVALUACIÓN
# =========================================================

@torch.no_grad()
def _evaluate_pytorch_classifier(model, loader, loss_fn, device):
    model.eval()

    running_loss = 0.0
    n_samples = 0

    all_probs = []
    all_preds = []
    all_true = []

    for xb, yb in loader:
        xb = xb.to(device=device, dtype=torch.float32)
        yb = yb.to(device=device, dtype=torch.float32)

        outputs = model(xb)
        probs_2c = outputs["out_activation"]

        loss = loss_fn(outputs, yb)

        probs_pos = probs_2c[:, 1]
        preds = torch.argmax(probs_2c, dim=1)
        y_true = torch.argmax(yb, dim=1)

        batch_size_curr = xb.size(0)
        running_loss += loss.item() * batch_size_curr
        n_samples += batch_size_curr

        all_probs.append(probs_pos.detach().cpu().numpy())
        all_preds.append(preds.detach().cpu().numpy())
        all_true.append(y_true.detach().cpu().numpy())

    y_prob = np.concatenate(all_probs).astype(np.float32)
    y_pred = np.concatenate(all_preds).astype(np.int64)
    y_true = np.concatenate(all_true).astype(np.int64)

    mean_loss = running_loss / max(n_samples, 1)

    try:
        auc_value = roc_auc_score(y_true, y_prob)
    except Exception:
        auc_value = np.nan

    metrics = {
        "loss": mean_loss,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred, average="binary", zero_division=0),
        "precision": precision_score(y_true, y_pred, average="binary", zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "auc": auc_value,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }

    return metrics

In [12]:
# =========================================================
# ENTRENAMIENTO CV POR SUJETOS PARA EEGNet - PARÁMETROS FIJOS
# =========================================================

def train_L24O_cv_fixed(
    model_name,
    X,
    y,
    sbjs,
    folds,
    model_args,
    compile_cfg,
    epochs=100,
    batch_size=16,
    seed=42,
    save_root="results_eegnet_fixed",
    run_name="EEGNet_fixed",
    cache_model_format="weights",
    force_retrain=True,
    evaluate_test=True,
):
    all_fold_metrics = []
    all_fold_val_metrics = []
    total_histories = []
    saved_model_paths = {}

    model_name = model_name.lower()

    if model_name != "eegnet":
        raise ValueError("Esta versión solo soporta 'EEGNet'.")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    run_dir = os.path.join(save_root, run_name)
    os.makedirs(run_dir, exist_ok=True)

    for fold, (train_subjects, val_subjects, test_subjects) in enumerate(folds):

        print("\n" + "=" * 80)
        print(f"FOLD {fold}")
        print("=" * 80)

        fold_dir = os.path.join(run_dir, f"fold_{fold}")
        os.makedirs(fold_dir, exist_ok=True)

        checkpoint_path = os.path.join(fold_dir, "resume_checkpoint.pt")
        best_state_path = os.path.join(fold_dir, "best_state.pt")
        fold_info_path = os.path.join(fold_dir, "fold_result.pkl")
        history_path = os.path.join(fold_dir, "history.pkl")

        if cache_model_format == "weights":
            cached_model_path = os.path.join(fold_dir, "final_state_dict.pt")
        elif cache_model_format == "full":
            cached_model_path = os.path.join(fold_dir, "final_model.pt")
        else:
            raise ValueError("cache_model_format debe ser 'weights' o 'full'")

        # =====================================================
        # Si force_retrain=True, borrar caché del fold
        # =====================================================
        if force_retrain:
            for p in [
                checkpoint_path,
                best_state_path,
                fold_info_path,
                history_path,
                cached_model_path,
            ]:
                if os.path.exists(p):
                    os.remove(p)

        # =====================================================
        # Reutilizar fold si ya existe y es compatible
        # =====================================================
        if (not force_retrain) and os.path.exists(fold_info_path) and os.path.exists(cached_model_path):
            with open(fold_info_path, "rb") as f:
                fold_info = pickle.load(f)

            cached_eval_mode = fold_info.get("evaluate_test", None)

            if cached_eval_mode == evaluate_test:
                if evaluate_test:
                    all_fold_metrics.append(fold_info["fold_metrics"])

                all_fold_val_metrics.append(fold_info["fold_val_metrics"])
                total_histories.append(fold_info["history"])
                saved_model_paths[fold] = cached_model_path

                print(f"Fold {fold}: reutilizado desde disco.")
                continue

            else:
                print(
                    f"Fold {fold}: caché incompatible "
                    f"(cached evaluate_test={cached_eval_mode}, actual={evaluate_test}). Reentrenando."
                )

        # =====================================================
        # Índices por sujeto
        # =====================================================
        train_idx = [i for i, sbj in enumerate(sbjs) if sbj in train_subjects]
        val_idx = [i for i, sbj in enumerate(sbjs) if sbj in val_subjects]
        test_idx = [i for i, sbj in enumerate(sbjs) if sbj in test_subjects]

        X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
        y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

        # =====================================================
        # Convertir y a one-hot aquí, sin tocar get_segmented_data()
        # =====================================================
        y_train = ensure_onehot_labels(y_train)
        y_val = ensure_onehot_labels(y_val)

        if evaluate_test:
            y_test = ensure_onehot_labels(y_test)

        print("Train:", X_train.shape, y_train.shape, "subjects:", len(set(np.array(sbjs)[train_idx])))
        print("Val  :", X_val.shape, y_val.shape, "subjects:", len(set(np.array(sbjs)[val_idx])))
        print("Test :", X_test.shape, y_test.shape if evaluate_test else None, "subjects:", len(set(np.array(sbjs)[test_idx])))

        # =====================================================
        # Semilla específica por fold
        # =====================================================
        set_seed(seed + fold)

        model_args_local = deepcopy(model_args)
        model_args_local["seed"] = seed + fold

        model = build_eeg_model(
            model_name=model_name,
            **model_args_local
        ).to(device)

        compile_cfg_local = deepcopy(compile_cfg)

        compile_args_local, callbacks = build_compile_config(
            model_name=model_name,
            total_epochs=epochs,
            **compile_cfg_local
        )

        loss_fn = compile_args_local["loss_fn"]

        optimizer, scheduler = _build_optimizer_and_scheduler(
            model,
            compile_args_local
        )

        loss_weights = compile_args_local.get("loss_weights", None)

        # =====================================================
        # Datasets y loaders
        # =====================================================
        train_dataset = torch.utils.data.TensorDataset(
            torch.tensor(X_train, dtype=torch.float32),
            torch.tensor(y_train, dtype=torch.float32),
        )

        val_dataset = torch.utils.data.TensorDataset(
            torch.tensor(X_val, dtype=torch.float32),
            torch.tensor(y_val, dtype=torch.float32),
        )

        g = torch.Generator()
        g.manual_seed(seed + fold)

        train_loader = torch.utils.data.DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            drop_last=False,
            generator=g,
        )

        val_loader = torch.utils.data.DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
        )

        test_loader = None

        if evaluate_test:
            test_dataset = torch.utils.data.TensorDataset(
                torch.tensor(X_test, dtype=torch.float32),
                torch.tensor(y_test, dtype=torch.float32),
            )

            test_loader = torch.utils.data.DataLoader(
                test_dataset,
                batch_size=batch_size,
                shuffle=False,
                drop_last=False,
            )

        # =====================================================
        # Variables de entrenamiento
        # =====================================================
        start_epoch = 0
        best_val_loss = np.inf
        patience_counter = 0

        history = {
            "epoch": [],
            "train_loss": [],
            "val_loss": [],
            "val_accuracy": [],
            "val_balanced_accuracy": [],
            "val_recall": [],
            "val_precision": [],
            "val_kappa": [],
            "val_auc": [],
            "lr": [],
        }

        # =====================================================
        # Reanudar si hay checkpoint compatible
        # =====================================================
        if (not force_retrain) and os.path.exists(checkpoint_path):
            ckpt = torch.load(checkpoint_path, map_location=device)

            model.load_state_dict(ckpt["model_state_dict"])
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])

            if scheduler is not None and ckpt.get("scheduler_state_dict", None) is not None:
                scheduler.load_state_dict(ckpt["scheduler_state_dict"])

            start_epoch = ckpt["epoch"] + 1
            best_val_loss = ckpt["best_val_loss"]
            patience_counter = ckpt["patience_counter"]
            history = ckpt["history"]

            print(f"Fold {fold}: reanudando desde epoch {start_epoch}.")

        # =====================================================
        # Loop de entrenamiento
        # =====================================================
        for epoch in range(start_epoch, epochs):

            for cb in callbacks:
                if hasattr(cb, "on_epoch_begin") and loss_weights is not None:
                    cb.on_epoch_begin(epoch, loss_weights)

            train_loss = _run_epoch_pytorch(
                model=model,
                loader=train_loader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                device=device,
            )

            val_eval = _evaluate_pytorch_classifier(
                model=model,
                loader=val_loader,
                loss_fn=loss_fn,
                device=device,
            )

            val_loss = val_eval["loss"]

            if scheduler is not None:
                scheduler.step(val_loss)

            current_lr = optimizer.param_groups[0]["lr"]

            history["epoch"].append(epoch + 1)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["val_accuracy"].append(val_eval["accuracy"])
            history["val_balanced_accuracy"].append(val_eval["balanced_accuracy"])
            history["val_recall"].append(val_eval["recall"])
            history["val_precision"].append(val_eval["precision"])
            history["val_kappa"].append(val_eval["kappa"])
            history["val_auc"].append(val_eval["auc"])
            history["lr"].append(current_lr)

            improved = val_loss < (best_val_loss - 1e-4)

            if improved:
                best_val_loss = val_loss
                patience_counter = 0
                torch.save(model.state_dict(), best_state_path)
            else:
                patience_counter += 1

            checkpoint = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
                "best_val_loss": best_val_loss,
                "patience_counter": patience_counter,
                "history": history,
            }

            torch.save(checkpoint, checkpoint_path)

            with open(history_path, "wb") as f:
                pickle.dump(history, f)

            if (epoch + 1) % 10 == 0:
                print(
                    f"Fold {fold} | Epoch {epoch + 1:03d} | "
                    f"train_loss={train_loss:.4f} | "
                    f"val_loss={val_loss:.4f} | "
                    f"val_acc={val_eval['accuracy']:.4f} | "
                    f"val_bal_acc={val_eval['balanced_accuracy']:.4f}"
                )

            if patience_counter >= 25:
                print(f"Early stopping en fold {fold}, epoch {epoch + 1}.")
                break

        # =====================================================
        # Cargar mejor estado del fold
        # =====================================================
        if os.path.exists(best_state_path):
            best_state = torch.load(best_state_path, map_location=device)
            model.load_state_dict(best_state)

        # =====================================================
        # Evaluación final en validación
        # =====================================================
        val_eval = _evaluate_pytorch_classifier(
            model=model,
            loader=val_loader,
            loss_fn=loss_fn,
            device=device,
        )

        fold_val_metrics = {
            "val_accuracy": val_eval["accuracy"],
            "val_balanced_accuracy": val_eval["balanced_accuracy"],
            "val_recall": val_eval["recall"],
            "val_precision": val_eval["precision"],
            "val_kappa": val_eval["kappa"],
            "val_auc": val_eval["auc"],
        }

        # =====================================================
        # Evaluación en test
        # =====================================================
        if evaluate_test:
            test_eval = _evaluate_pytorch_classifier(
                model=model,
                loader=test_loader,
                loss_fn=loss_fn,
                device=device,
            )

            fold_metrics = {
                "accuracy": test_eval["accuracy"],
                "balanced_accuracy": test_eval["balanced_accuracy"],
                "recall": test_eval["recall"],
                "precision": test_eval["precision"],
                "kappa": test_eval["kappa"],
                "auc": test_eval["auc"],
            }

            print(
                f"Fold {fold} TEST | "
                f"acc={fold_metrics['accuracy']:.4f} | "
                f"bal_acc={fold_metrics['balanced_accuracy']:.4f} | "
                f"auc={fold_metrics['auc']:.4f}"
            )

        else:
            fold_metrics = None

        # =====================================================
        # Guardar modelo final del fold
        # =====================================================
        if cache_model_format == "weights":
            torch.save(model.state_dict(), cached_model_path)
        else:
            torch.save(model, cached_model_path)

        fold_info = {
            "fold_metrics": fold_metrics,
            "fold_val_metrics": fold_val_metrics,
            "history": history,
            "evaluate_test": evaluate_test,
            "model_args": model_args,
            "compile_cfg": compile_cfg,
        }

        with open(fold_info_path, "wb") as f:
            pickle.dump(fold_info, f)

        if evaluate_test:
            all_fold_metrics.append(fold_metrics)

        all_fold_val_metrics.append(fold_val_metrics)
        total_histories.append(history)
        saved_model_paths[fold] = cached_model_path

        print(f"Fold {fold}: entrenado y guardado.")

    # =========================================================
    # Promedios finales
    # =========================================================
    mean_metrics = None
    mean_accuracy = None

    if evaluate_test and len(all_fold_metrics) > 0:
        mean_metrics = {}

        for key in all_fold_metrics[0].keys():
            vals = [f[key] for f in all_fold_metrics]
            mean_metrics[f"mean_{key}"] = float(np.nanmean(vals))
            mean_metrics[f"std_{key}"] = float(np.nanstd(vals))

        accs_general = [f["accuracy"] for f in all_fold_metrics]
        mean_accuracy = float(np.nanmean(accs_general))

    mean_val_metrics = {}

    for key in all_fold_val_metrics[0].keys():
        vals = [f[key] for f in all_fold_val_metrics]
        mean_val_metrics[f"mean_{key}"] = float(np.nanmean(vals))
        mean_val_metrics[f"std_{key}"] = float(np.nanstd(vals))

    val_accs_general = [f["val_accuracy"] for f in all_fold_val_metrics]
    val_bal_accs_general = [f["val_balanced_accuracy"] for f in all_fold_val_metrics]

    results = {
        "mean_accuracy": mean_accuracy,
        "mean_val_accuracy": float(np.nanmean(val_accs_general)),
        "mean_val_balanced_accuracy": float(np.nanmean(val_bal_accs_general)),
        "fold_metrics": all_fold_metrics if evaluate_test else None,
        "fold_val_metrics": all_fold_val_metrics,
        "mean_metrics": mean_metrics,
        "mean_val_metrics": mean_val_metrics,
        "histories": total_histories,
        "saved_model_paths": saved_model_paths,
        "run_dir": run_dir,
    }

    # Guardar resumen global
    summary_path = os.path.join(run_dir, "summary_results.pkl")
    with open(summary_path, "wb") as f:
        pickle.dump(results, f)

    print("\n" + "=" * 80)
    print("RESULTADOS PROMEDIO")
    print("=" * 80)

    print("Validación:")
    print(f"  mean_val_accuracy          : {results['mean_val_accuracy']:.4f}")
    print(f"  mean_val_balanced_accuracy : {results['mean_val_balanced_accuracy']:.4f}")

    if evaluate_test and results["mean_metrics"] is not None:
        print("\nTest:")
        for k, v in results["mean_metrics"].items():
            print(f"  {k}: {v:.4f}")

    print(f"\nResultados guardados en: {run_dir}")

    return results

In [13]:
import pickle
import os
import json
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# =========================================================
# CONFIGURACIÓN GENERAL
# =========================================================

model_name = "EEGNet"
db = "TDAH"
seed = 42

epochs = 100
batch_size = 16
force_retrain = True
save_format = "weights"

# =========================================================
# RUTA DIRECTA DE GUARDADO
# =========================================================

SAVE_ROOT = "/home/usuario-utp/Desktop/Yessica Alejandra/resultados_eegnet_tdah_fixed_seed42"
os.makedirs(SAVE_ROOT, exist_ok=True)

run_name = f"{model_name}_{db}_fixed_seed{seed}"

RESULTS_JSON = os.path.join(SAVE_ROOT, "summary_results.json")
RESULTS_PKL = os.path.join(SAVE_ROOT, "summary_results.pkl")

# =========================================================
# CARGAR FOLDS
# =========================================================

with open(
    "/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/folds.pkl",
    "rb"
) as f:
    folds = pickle.load(f)

# =========================================================
# CARGAR DATA
# =========================================================

X, y, sbjs, _ = get_segmented_data()

print("=" * 80)
print("DATA CARGADA")
print("=" * 80)
print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)
print("Número de sujetos:", len(set(sbjs)))
print("Número de folds:", len(folds))

# =========================================================
# PARÁMETROS FIJOS EEGNet
# =========================================================

model_args = {
    "Chans": 19,
    "Samples": 512,
    "nb_classes": 2,
    "dropoutRate": 0.5,
    "kernLength": 32,
    "F1": 8,
    "D": 2,
    "F2": 16,
    "norm_rate": 0.25,
    "dropoutType": "Dropout",
}

compile_cfg = {
    "learning_rate": 1e-2,
}

# =========================================================
# ENTRENAR EEGNet FIJO
# =========================================================

results = train_L24O_cv_fixed(
    model_name=model_name,
    X=X,
    y=y,
    sbjs=sbjs,
    folds=folds,
    model_args=model_args,
    compile_cfg=compile_cfg,
    epochs=epochs,
    batch_size=batch_size,
    seed=seed,
    save_root=SAVE_ROOT,
    run_name=run_name,
    cache_model_format=save_format,
    force_retrain=force_retrain,
    evaluate_test=True,
)

# =========================================================
# GUARDAR RESULTADOS GLOBALES
# =========================================================

with open(RESULTS_PKL, "wb") as f:
    pickle.dump(results, f)

# Convertir a JSON solo lo serializable
json_results = {
    "model_name": model_name,
    "db": db,
    "seed": seed,
    "epochs": epochs,
    "batch_size": batch_size,
    "model_args": model_args,
    "compile_cfg": compile_cfg,
    "mean_accuracy": results.get("mean_accuracy"),
    "mean_val_accuracy": results.get("mean_val_accuracy"),
    "mean_val_balanced_accuracy": results.get("mean_val_balanced_accuracy"),
    "mean_metrics": results.get("mean_metrics"),
    "mean_val_metrics": results.get("mean_val_metrics"),
    "run_dir": results.get("run_dir"),
}

with open(RESULTS_JSON, "w") as f:
    json.dump(json_results, f, indent=4)

# =========================================================
# RESUMEN FINAL
# =========================================================

print("\n" + "=" * 80)
print("ENTRENAMIENTO FINALIZADO - EEGNet FIJO")
print("=" * 80)
print(f"Carpeta principal       : {SAVE_ROOT}")
print(f"Run name                : {run_name}")
print(f"Seed                    : {seed}")
print(f"Resultados PKL          : {RESULTS_PKL}")
print(f"Resultados JSON         : {RESULTS_JSON}")

print("\nValidación:")
print(f"  mean_val_accuracy          : {results['mean_val_accuracy']:.4f}")
print(f"  mean_val_balanced_accuracy : {results['mean_val_balanced_accuracy']:.4f}")

if results["mean_metrics"] is not None:
    print("\nTest:")
    for k, v in results["mean_metrics"].items():
        print(f"  {k}: {v:.4f}")

Sujeto eliminado explícitamente: v36p
Sujetos TDAH encontrados: 60
Sujetos Control encontrados: 60
Sujetos totales encontrados: 120


DATA CARGADA
X: (8213, 19, 512) float32
y: (8213, 2) float32
Número de sujetos: 120
Número de folds: 5

FOLD 0
Train: (4977, 19, 512) (4977, 2) subjects: 76
Val  : (1574, 19, 512) (1574, 2) subjects: 20
Test : (1662, 19, 512) (1662, 2) subjects: 24
Fold 0 | Epoch 010 | train_loss=0.0833 | val_loss=1.0103 | val_acc=0.7490 | val_bal_acc=0.7319
Fold 0 | Epoch 020 | train_loss=0.0635 | val_loss=1.5055 | val_acc=0.7452 | val_bal_acc=0.7280
Fold 0 | Epoch 030 | train_loss=0.0519 | val_loss=2.0113 | val_acc=0.6798 | val_bal_acc=0.6579
Fold 0 | Epoch 040 | train_loss=0.0689 | val_loss=2.7066 | val_acc=0.4041 | val_bal_acc=0.4376
Early stopping en fold 0, epoch 40.
Fold 0 TEST | acc=0.8694 | bal_acc=0.8544 | auc=0.9909
Fold 0: entrenado y guardado.

FOLD 1
Train: (5223, 19, 512) (5223, 2) subjects: 76
Val  : (1428, 19, 512) (1428, 2) subjects: 20
Test : (1562, 19, 512) (1562, 2) subjects: 24
Fold 1 | Epoch 010 | train_loss=0.0835 | val_loss=0.3564 | val_acc=0.8634 | val_bal_acc=0.8557
Fold 1 | 

In [14]:
import os
import json
import pickle
import random
import warnings
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

# =========================================================
# 1) CONFIGURACIÓN GENERAL
# =========================================================
model_name = "EEGNet"
db = "TDAH"

SAVE_ROOT = "/home/usuario-utp/Desktop/Yessica Alejandra/resultados_eegnet_tdah_repeated"
os.makedirs(SAVE_ROOT, exist_ok=True)

RESULTS_CSV = os.path.join(SAVE_ROOT, "repeated_test_results.csv")
RESULTS_JSON = os.path.join(SAVE_ROOT, "repeated_test_summary.json")

# 10 repeticiones -> cada seed base corre los 5 folds
seeds = list(range(10))

epochs = 100
batch_size = 16
force_retrain = True
save_format = "weights"

# =========================================================
# 2) PARÁMETROS FIJOS EEGNet
# =========================================================
model_args = {
    "Chans": 19,
    "Samples": 512,
    "nb_classes": 2,
    "dropoutRate": 0.5,
    "kernLength": 32,
    "F1": 8,
    "D": 2,
    "F2": 16,
    "norm_rate": 0.25,
    "dropoutType": "Dropout",
}

compile_cfg = {
    "learning_rate": 1e-2,
}

# =========================================================
# 3) FUNCIONES AUXILIARES
# =========================================================
def set_all_seeds(seed: int):
    """
    Seed base de cada repetición.
    Dentro de train_L24O_cv_fixed se usará seed + fold,
    manteniendo la lógica anterior.
    """
    set_seed(seed)


def summarize_metric(values):
    arr = np.array(values, dtype=float)
    return {
        "mean": float(np.nanmean(arr)),
        "std_population": float(np.nanstd(arr, ddof=0)),
        "std_sample": float(np.nanstd(arr, ddof=1)) if len(arr) > 1 else 0.0,
        "n": int(len(arr)),
    }


# =========================================================
# 4) CARGAR DATOS Y FOLDS
# =========================================================
with open(
    "/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/folds.pkl",
    "rb"
) as f:
    folds = pickle.load(f)

X, y, sbjs, _ = get_segmented_data()

print("=" * 90)
print("CONFIGURACIÓN FIJA USADA EN LAS REPETICIONES - EEGNet")
print("=" * 90)

print("model_args:")
for k, v in model_args.items():
    print(f"  {k}: {v}")

print("\ncompile_cfg:")
for k, v in compile_cfg.items():
    print(f"  {k}: {v}")

print("\nDatos:")
print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)
print("Sujetos:", len(set(sbjs)))
print("Folds:", len(folds))


# =========================================================
# 5) REPETICIONES
#    Cada seed base corre los 5 folds.
#    Dentro de cada fold se usa seed + fold.
# =========================================================
all_rows = []
all_run_summaries = []

for rep_id, seed in enumerate(seeds):
    print("\n" + "=" * 90)
    print(f"REPETICIÓN {rep_id + 1}/{len(seeds)} | seed base = {seed}")
    print("=" * 90)

    set_all_seeds(seed)

    run_name = f"{model_name}_{db}_fixed_seed_{seed}"

    results = train_L24O_cv_fixed(
        model_name=model_name,
        X=X,
        y=y,
        sbjs=sbjs,
        folds=folds,
        model_args=model_args,
        compile_cfg=compile_cfg,
        epochs=epochs,
        batch_size=batch_size,
        seed=seed,
        save_root=SAVE_ROOT,
        run_name=run_name,
        cache_model_format=save_format,
        force_retrain=force_retrain,
        evaluate_test=True,
    )

    fold_metrics = results["fold_metrics"]

    for fold_id, fm in enumerate(fold_metrics):
        row = {
            "seed": seed,
            "repeat_id": rep_id,
            "fold": fold_id,
            "accuracy": float(fm["accuracy"]),
            "balanced_accuracy": float(fm["balanced_accuracy"]),
            "recall": float(fm["recall"]),
            "precision": float(fm["precision"]),
            "kappa": float(fm["kappa"]),
            "auc": float(fm["auc"]),
        }
        all_rows.append(row)

    run_summary = {
        "seed": seed,
        "repeat_id": rep_id,
        "mean_accuracy_across_5_folds": float(results["mean_accuracy"]),
        "mean_metrics": {
            k: float(v) for k, v in results["mean_metrics"].items()
        },
        "run_dir": results["run_dir"],
    }

    all_run_summaries.append(run_summary)


# =========================================================
# 6) GUARDAR RESULTADOS CRUDOS
# =========================================================
df = pd.DataFrame(all_rows)
df.to_csv(RESULTS_CSV, index=False)

print("\nArchivo CSV guardado en:")
print(RESULTS_CSV)


# =========================================================
# 7) RESUMEN GLOBAL SOBRE LAS 50 CORRIDAS
# =========================================================
summary_global = {
    "accuracy": summarize_metric(df["accuracy"].tolist()),
    "balanced_accuracy": summarize_metric(df["balanced_accuracy"].tolist()),
    "recall": summarize_metric(df["recall"].tolist()),
    "precision": summarize_metric(df["precision"].tolist()),
    "kappa": summarize_metric(df["kappa"].tolist()),
    "auc": summarize_metric(df["auc"].tolist()),
}


# =========================================================
# 8) RESUMEN POR FOLD
# =========================================================
summary_by_fold = {}

for fold_id in sorted(df["fold"].unique()):
    dff = df[df["fold"] == fold_id]

    summary_by_fold[f"fold_{fold_id}"] = {
        "accuracy": summarize_metric(dff["accuracy"].tolist()),
        "balanced_accuracy": summarize_metric(dff["balanced_accuracy"].tolist()),
        "recall": summarize_metric(dff["recall"].tolist()),
        "precision": summarize_metric(dff["precision"].tolist()),
        "kappa": summarize_metric(dff["kappa"].tolist()),
        "auc": summarize_metric(dff["auc"].tolist()),
    }


# =========================================================
# 9) RESUMEN POR SEED
# =========================================================
summary_by_seed = {}

for seed_val in sorted(df["seed"].unique()):
    dff = df[df["seed"] == seed_val]

    summary_by_seed[f"seed_{seed_val}"] = {
        "accuracy": summarize_metric(dff["accuracy"].tolist()),
        "balanced_accuracy": summarize_metric(dff["balanced_accuracy"].tolist()),
        "recall": summarize_metric(dff["recall"].tolist()),
        "precision": summarize_metric(dff["precision"].tolist()),
        "kappa": summarize_metric(dff["kappa"].tolist()),
        "auc": summarize_metric(dff["auc"].tolist()),
    }


# =========================================================
# 10) GUARDAR JSON FINAL
# =========================================================
final_summary = {
    "model_name": model_name,
    "database": db,
    "model_args": model_args,
    "compile_cfg": compile_cfg,
    "n_folds": 5,
    "n_repetitions": len(seeds),
    "total_test_runs": int(len(df)),
    "global_summary_over_50_runs": summary_global,
    "summary_by_fold": summary_by_fold,
    "summary_by_seed": summary_by_seed,
    "run_summaries": all_run_summaries,
    "csv_path": RESULTS_CSV,
}

with open(RESULTS_JSON, "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=4)

print("\nArchivo JSON guardado en:")
print(RESULTS_JSON)


# =========================================================
# 11) IMPRIMIR RESULTADOS FINALES
# =========================================================
print("\n" + "=" * 90)
print("RESULTADOS FINALES - EEGNet - 5 FOLDS x 10 REPETICIONES = 50 TESTS")
print("=" * 90)

print(f"Accuracy          : {summary_global['accuracy']['mean']:.4f} ± {summary_global['accuracy']['std_sample']:.4f}")
print(f"Balanced Accuracy : {summary_global['balanced_accuracy']['mean']:.4f} ± {summary_global['balanced_accuracy']['std_sample']:.4f}")
print(f"Recall            : {summary_global['recall']['mean']:.4f} ± {summary_global['recall']['std_sample']:.4f}")
print(f"Precision         : {summary_global['precision']['mean']:.4f} ± {summary_global['precision']['std_sample']:.4f}")
print(f"Kappa             : {summary_global['kappa']['mean']:.4f} ± {summary_global['kappa']['std_sample']:.4f}")
print(f"AUC               : {summary_global['auc']['mean']:.4f} ± {summary_global['auc']['std_sample']:.4f}")

print("\nResumen por fold:")
for fold_name, vals in summary_by_fold.items():
    print(
        f"{fold_name}: "
        f"Acc = {vals['accuracy']['mean']:.4f} ± {vals['accuracy']['std_sample']:.4f} | "
        f"Bal Acc = {vals['balanced_accuracy']['mean']:.4f} ± {vals['balanced_accuracy']['std_sample']:.4f}"
    )

print("\nResumen por seed:")
for seed_name, vals in summary_by_seed.items():
    print(
        f"{seed_name}: "
        f"Acc = {vals['accuracy']['mean']:.4f} ± {vals['accuracy']['std_sample']:.4f} | "
        f"Bal Acc = {vals['balanced_accuracy']['mean']:.4f} ± {vals['balanced_accuracy']['std_sample']:.4f}"
    )

Sujeto eliminado explícitamente: v36p
Sujetos TDAH encontrados: 60
Sujetos Control encontrados: 60
Sujetos totales encontrados: 120
CONFIGURACIÓN FIJA USADA EN LAS REPETICIONES - EEGNet
model_args:
  Chans: 19
  Samples: 512
  nb_classes: 2
  dropoutRate: 0.5
  kernLength: 32
  F1: 8
  D: 2
  F2: 16
  norm_rate: 0.25
  dropoutType: Dropout

compile_cfg:
  learning_rate: 0.01

Datos:
X: (8213, 19, 512) float32
y: (8213, 2) float32
Sujetos: 120
Folds: 5

REPETICIÓN 1/10 | seed base = 0

FOLD 0
Train: (4977, 19, 512) (4977, 2) subjects: 76
Val  : (1574, 19, 512) (1574, 2) subjects: 20
Test : (1662, 19, 512) (1662, 2) subjects: 24
Fold 0 | Epoch 010 | train_loss=0.1015 | val_loss=0.8213 | val_acc=0.7764 | val_bal_acc=0.7439
Fold 0 | Epoch 020 | train_loss=0.0659 | val_loss=0.6140 | val_acc=0.8094 | val_bal_acc=0.8060
Early stopping en fold 0, epoch 29.
Fold 0 TEST | acc=0.7996 | bal_acc=0.7955 | auc=0.9193
Fold 0: entrenado y guardado.

FOLD 1
Train: (5223, 19, 512) (5223, 2) subjects: 76
